### Configuración inicial

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
import os

# Configuración del Master y Delta
master_url = "spark://spark-master:7077"

builder = SparkSession.builder \
    .appName("Ingesta_Bronze_SECOP") \
    .master(master_url) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "4") # Ajustado para entorno local

# Inicializar Spark con soporte para Delta Lake
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada con éxito")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-913268bc-5cb6-4bb7-91b9-e2d3da403529;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 189ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

SparkSession iniciada con éxito


### Lectura del archivo CSV 

In [2]:
csv_path = "/app/data/SECOP_II_Contratos_Electronicos_20260126.csv"

print("Leyendo CSV ..")

df_raw = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .load(csv_path)

print(f"Total de registros leídos: {df_raw.count()}")
df_raw.limit(5).toPandas()

Leyendo CSV ..


26/01/28 21:27:25 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

Total de registros leídos: 5276398


26/01/28 21:47:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,Nombre Entidad,Nit Entidad,Departamento,Ciudad,Localización,Orden,Sector,Rama,Entidad Centralizada,Proceso de Compra,...,Tipo de documento Ordenador del gasto,Número de documento Ordenador del gasto,Nombre supervisor,Tipo de documento supervisor,Número de documento supervisor,Nombre Ordenador de Pago,Tipo de documento Ordenador de Pago,Número de documento Ordenador de Pago,Documentos Tipo,Descripcion Documentos Tipo
0,JEP,901.140.004,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",Nacional,No aplica/No pertenece,Corporación Autónoma,Centralizada,CO1.BDOS.1108279,...,No definido,No definido,No definido,No definido,No definido,No definido,No definido,No definido,No,No definido
1,ALCALDIA MUNICIPIO DE ARAUCA,800.102.504,Arauca,Arauca,"Colombia, Arauca , Arauca",Territorial,Servicio Público,Ejecutivo,Descentralizada,CO1.BDOS.3106524,...,No definido,No definido,No definido,No definido,No definido,No definido,No definido,No definido,No,No definido
2,RADIO TELEVISION NACIONAL DE COLOMBIA.,900.002.583,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",Nacional,Tecnologías de la Información y las Comunicaci...,Ejecutivo,Centralizada,CO1.BDOS.4132503,...,No definido,No definido,No definido,No definido,No definido,No definido,No definido,No definido,No,No definido
3,ALCALDIA DE GUAMAL - META,800.098.193,Meta,Guamal,"Colombia, Meta , Guamal",Territorial,No aplica/No pertenece,Corporación Autónoma,Centralizada,CO1.BDOS.7927329,...,Cédula de Ciudadanía,86081881,GILMA CONSUELO ROBAYO MOYA,Cédula de Ciudadanía,51976005,No definido,No definido,No definido,No,No definido
4,ALCALDIA MUNICIPAL DE VALLEDUPAR,800.098.911,Cesar,Valledupar,"Colombia, Cesar , Valledupar",Territorial,Servicio Público,Ejecutivo,Centralizada,CO1.BDOS.2250151,...,Cédula de Ciudadanía,1065578810,LUIS ENRIQUE GALVIS NUÑEZ,Cédula de Ciudadanía,77186812,No definido,No definido,No definido,No,No definido


### Limpiar nombres de columnas

In [3]:
df_bronze = df_raw

for col_name in df_bronze.columns:
    clean_name = col_name.lower() \
        .replace(" ", "_") \
        .replace(".", "") \
        .replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u") \
        .replace("(", "").replace(")", "") \
        .replace(",", "").replace(";", "") # Añadimos estos para evitar el error de Delta
    
    df_bronze = df_bronze.withColumnRenamed(col_name, clean_name)

# 2. AGREGAR METADATA DE AUDITORÍA
df_bronze = df_bronze.withColumn(
    "_ingestion_time", 
    F.current_timestamp()
).withColumn(
    "_source_file", 
    F.input_file_name()
)

print("Columnas normalizadas y datos de auditoría agregados.")
print(f"Nuevas columnas: {df_bronze.columns[:5]}...") 
df_bronze.select("nombre_entidad", "_ingestion_time", "_source_file").show(5, False)

Columnas normalizadas y datos de auditoría agregados.
Nuevas columnas: ['nombre_entidad', 'nit_entidad', 'departamento', 'ciudad', 'localizacion']...
+--------------------------------------+-------------------------+-------------------------------------------------------------+
|nombre_entidad                        |_ingestion_time          |_source_file                                                 |
+--------------------------------------+-------------------------+-------------------------------------------------------------+
|JEP                                   |2026-01-28 21:47:20.68063|file:///app/data/SECOP_II_Contratos_Electronicos_20260126.csv|
|ALCALDIA MUNICIPIO DE ARAUCA          |2026-01-28 21:47:20.68063|file:///app/data/SECOP_II_Contratos_Electronicos_20260126.csv|
|RADIO TELEVISION NACIONAL DE COLOMBIA.|2026-01-28 21:47:20.68063|file:///app/data/SECOP_II_Contratos_Electronicos_20260126.csv|
|ALCALDIA DE GUAMAL - META             |2026-01-28 21:47:20.68063|file:///ap

### Guardar datos en Delta

In [4]:
# Usamos una ruta absoluta desde la raíz del contenedor
output_path = "/app/data/lakehouse/bronze/secop"

print(f"Guardando en capa Bronce en la ruta: {output_path}...")

df_bronze.repartition(10).write.format("delta") \
    .mode("overwrite") \
    .save(output_path)

print(f" ¡Ingesta completada! Datos protegidos en Delta Lake.")
print(f" Total columnas en Bronce: {len(df_bronze.columns)}")

Guardando en capa Bronce en la ruta: /app/data/lakehouse/bronze/secop...


 ¡Ingesta completada! Datos protegidos en Delta Lake.
 Total columnas en Bronce: 89


### Leer Delta

In [5]:
bronze_path = "/app/data/lakehouse/bronze/secop"

# Leemos usando el formato "delta"
df_bronze_check = spark.read.format("delta").load(bronze_path)

print(f" Lectura exitosa. Registros en el Lakehouse: {df_bronze_check.count()}")
# Ver las primeras 5 filas con Pandas 
df_bronze_check.limit(5).toPandas()

 Lectura exitosa. Registros en el Lakehouse: 5276398


,nombre_entidad,nit_entidad,departamento,ciudad,localizacion,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,nombre_supervisor,tipo_de_documento_supervisor,numero_de_documento_supervisor,nombre_ordenador_de_pago,tipo_de_documento_ordenador_de_pago,numero_de_documento_ordenador_de_pago,documentos_tipo,descripcion_documentos_tipo,_ingestion_time,_source_file
0,SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR E.S...,900.958.564,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",Territorial,Salud y Protección Social,Ejecutivo,Descentralizada,CO1.BDOS.5110380,...,No definido,No definido,No definido,No definido,No definido,No definido,No,No definido,2026-01-28 21:47:22.516438,file:///app/data/SECOP_II_Contratos_Electronic...
1,AEROCIVIL,899.999.059,Distrito Capital de Bogotá,No Definido,"Colombia, Bogotá, No Definido",Nacional,Transporte,Ejecutivo,Centralizada,CO1.BDOS.7701331,...,VICTORIA EUGENIA MURILLO POLO,Cédula de Ciudadanía,38566824,No definido,No definido,No definido,No,No definido,2026-01-28 21:47:22.516438,file:///app/data/SECOP_II_Contratos_Electronic...
2,IDEP,830.007.738,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",Territorial,Educación Nacional,Ejecutivo,Descentralizada,CO1.BDOS.7859596,...,ANDRES RIOS LEON,Cédula de Ciudadanía,1023863290,No definido,No definido,No definido,No,No definido,2026-01-28 21:47:22.516438,file:///app/data/SECOP_II_Contratos_Electronic...
3,AGENCIA NACIONAL DE TIERRAS - ANT,900.948.953,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",Nacional,agricultura,Ejecutivo,Centralizada,CO1.BDOS.6693577,...,JESUS BAYRO MUÑOZ FELIX,Cédula de Ciudadanía,79380897,No definido,No definido,No definido,No,No definido,2026-01-28 21:47:22.516438,file:///app/data/SECOP_II_Contratos_Electronic...
4,GOBERNACION DEL CHOCÓ.,891.680.010,No Definido,No Definido,"Colombia, No Definido, No Definido",Territorial,No aplica/No pertenece,Ejecutivo,Descentralizada,CO1.BDOS.4868797,...,JHALMER AUGUSTO LONDOÑO PALACIOS,Cédula de Ciudadanía,71725187,ARIEL PALACIOS CALDERON,Cédula de Ciudadanía,71974534,No,No definido,2026-01-28 21:47:22.516438,file:///app/data/SECOP_II_Contratos_Electronic...
